# Работа №15. Компьютерное зрение с использованием свёрточных нейронных сетей

**Вариант 1** (табл. 15.2): `epoch = 4`, `k = 3`, `out = 16`, `pad = 1`.
Размер изображений MNIST: **14×14** (режим «Google Colab / слабый ПК»). Для датасетов в разделах 3–5 выбран датасет **«Настроение клиентов по лицам»** (64×64, число эпох = 4 × n, n = 3).

## 1. Установка библиотек
`pip install numpy pandas pillow scikit-learn matplotlib` (ячейка удалена после выполнения, п. 1.2).

## 2. Классификация изображений MNIST свёрточной нейронной сетью
### Ячейка 1 — импорт библиотек

In [1]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import random, pathlib, os, urllib, zipfile, shutil, platform, requests, io
RND = 42
np.random.seed(RND)
random.seed(RND)
def last_root_folder(data_dir):
    out = ''
    for path, subfolders, _ in os.walk(data_dir):
        if subfolders==[]: return pathlib.Path(out).with_suffix('')
        else: out = path
def asfcont(x):
    x = np.asarray(x, dtype=np.float32)
    if not x.flags['C_CONTIGUOUS']:
        x = np.ascontiguousarray(x)
    return x
def im2col_batch_strides(X, k, stride=1, padding=0):
    X = asfcont(X)
    N, H, W, C = X.shape
    H_p = H + 2*padding
    W_p = W + 2*padding
    out_h = (H + 2*padding - k)//stride + 1
    out_w = (W + 2*padding - k)//stride + 1
    if padding > 0:
        Xp = np.pad(X,((0,0),(padding,padding),
        (padding,padding),(0,0)), mode='constant')
    else:
        Xp = X
    s0, s1, s2, s3 = Xp.strides
    shape = (N, out_h, out_w, k, k, C)
    strides = (s0, s1*stride, s2*stride, s1, s2, s3)
    windows = as_strided(Xp, shape=shape, strides=strides)
    N_, oh, ow, kk1, kk2, C_ = windows.shape
    cols = windows.reshape(N_, oh*ow, kk1*kk2*C_).transpose(0,2,1)
    return cols, out_h, out_w

### Ячейка 2 — преобразование набора изображений в батч и обратно

In [2]:
def col2im_batch_strides(dcols, X_shape, k, stride=1, padding=0):
    N, H, W, C = X_shape
    H_p = H + 2*padding
    W_p = W + 2*padding
    out_h = (H + 2*padding - k)//stride + 1
    out_w = (W + 2*padding - k)//stride + 1
    dcols_t = dcols.transpose(0,2,1).reshape(N, out_h, out_w, k, k, C)
    Xp = np.zeros((N, H_p, W_p, C), dtype=np.float32)
    for p in range(k):
        p_end = p + stride*out_h
        for q in range(k):
            q_end = q + stride*out_w
            Xp[:, p:p_end:stride, q:q_end:stride, :] += dcols_t[:, :, :, p, q, :]
    if padding > 0:
        return Xp[:, padding:-padding, padding:-padding, :]
    return Xp

### Ячейка 3 — функции активации ReLU и Softmax

In [3]:
def relu(x): return np.maximum(0, x)
def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

### Ячейка 4 — One-Hot кодирование классов

In [4]:
def to_one_hot(y, C):
    y = np.array(y).astype(int).ravel()
    oh = np.zeros((y.shape[0], C), dtype=np.float32)
    oh[np.arange(y.shape[0]), y] = 1.0
    return oh

### Ячейка 5 — свёрточный слой и слой подвыборки (пулинг)

In [5]:
def conv2d_batch_im2col(X, W, b, stride=1, padding=0):
    X = asfcont(X)
    W = asfcont(W)
    b = np.asarray(b, dtype=np.float32).reshape(-1)
    k = W.shape[0]
    C_in = W.shape[2]
    C_out = W.shape[3]
    cols, out_h, out_w = im2col_batch_strides(X, k, stride=stride, padding=padding)
    N, K, L = cols.shape
    assert K == k*k*C_in, f"K неверный: ждал {k*k*C_in}, получил {K}"
    filters = W.reshape(-1, C_out)
    out_nlc = np.tensordot(cols, filters, axes=([1],[0]))
    out = out_nlc.reshape(N, out_h, out_w, C_out).astype(np.float32)
    out += b.reshape(1,1,1,-1)
    return out, cols, out_h, out_w
def maxpool2d_batch(X, pool=2, stride=2, return_mask=False):
    X = asfcont(X)
    N, H, W, C = X.shape
    out_h = (H - pool)//stride + 1
    out_w = (W - pool)//stride + 1
    s0, s1, s2, s3 = X.strides
    shape = (N, out_h, out_w, pool, pool, C)
    strides = (s0, s1*stride, s2*stride, s1, s2, s3)
    windows = as_strided(X, shape=shape, strides=strides)
    wr = windows.reshape(N, out_h, out_w, pool*pool, C)
    max_vals = wr.max(axis=3)
    mask = (wr == max_vals[..., None, :]).astype(np.float32)
    out = max_vals
    if return_mask: return out, mask
    else: return out

### Ячейка 6 — архитектура свёрточной нейронной сети (инициализация параметров)

In [6]:
def init_cnn_params(input_channels,num_classes,seed=RND,
conv_layers=None,input_size=None):
    if conv_layers is None: conv_layers=[{'k':3,'out':16,'pad':1,'stride':1}]
    if input_size is None: h_current=img_height;w_current=img_width
    else: h_current,w_current=input_size
    params={};in_channels=input_channels
    for i,cfg in enumerate(conv_layers, start=1):
        k=cfg.get('k',3);pad=cfg.get('pad',0)
        stride_conv=cfg.get('stride',1);out_ch=cfg['out']
        np.random.seed(seed+i)
        W=np.random.randn(k,k,in_channels,out_ch).astype(np.float32)*0.05
        b=np.zeros((out_ch,),dtype=np.float32)
        params[f'W{i}']=W;params[f'b{i}']=b
        H_conv=(h_current+2*pad-k)//stride_conv+1
        W_conv=(w_current+2*pad-k)//stride_conv+1
        pool=2;pool_stride=2
        h_current=(H_conv-pool)//pool_stride+1
        w_current=(W_conv-pool)//pool_stride+1
        in_channels=out_ch
    flat_dim=h_current*w_current*in_channels
    if flat_dim<=0:
        raise ValueError(f"flat_dim={flat_dim}")
    W_final=np.random.randn(flat_dim,num_classes).astype(np.float32)*0.05
    b_final=np.zeros((num_classes,),dtype=np.float32)
    params[f'W{len(conv_layers)+1}']=W_final
    params[f'b{len(conv_layers)+1}']=b_final
    params['_conv_layers']=conv_layers;params['_input_size']=(h_current,w_current)
    return params

### Ячейка 7 — прямой проход через нейронную сеть

In [7]:
def forward_batch(Xb,params):
    cache={'Xb':asfcont(Xb)};output=asfcont(Xb)
    conv_layers_cfg=params.get('_conv_layers',None)
    if conv_layers_cfg is None:
        conv_layers=int(len([k for k in params.keys() if k.startswith('W')])/2)-1
        conv_layers_cfg=[{'k':None,'pad':1,'stride':1}]*conv_layers
    for i,cfg in enumerate(conv_layers_cfg,start=1):
        cache[f'input_shape{i}']=output.shape
        W=params[f'W{i}'];b=params[f'b{i}']
        pad=cfg.get('pad',0);stride_conv=cfg.get('stride',1)
        C_w=W.shape[2];C_x=output.shape[3]
        if C_x!=C_w:
            if C_w==1:
                output=np.mean(output,axis=3,keepdims=True).astype(np.float32)
            else:
                output=np.tile(output[..., :1],(1,1,1,C_w)).astype(np.float32)
        z_conv,cols,oh,ow=conv2d_batch_im2col(output,W,b,
        stride=stride_conv,padding=pad)
        a_relu=relu(z_conv)
        z_pool,mask=maxpool2d_batch(a_relu,pool=2,stride=2,return_mask=True)
        cache[f'cols{i}']=cols;cache[f'a{i}']=a_relu;cache[f'mask{i}']=mask
        output=z_pool
    flat=output.reshape(output.shape[0],-1)
    cache['pre_flat_shape']=output.shape;cache['output_shape']=output.shape
    z_mlp=flat.dot(asfcont(params[f'W{len(conv_layers_cfg)+1}']))
    z_mlp+=params[f'b{len(conv_layers_cfg)+1}'].reshape(1,-1)
    probs=softmax(z_mlp);cache['flat']=flat;cache['probs']=probs
    return probs,cache

### Ячейка 8 — шаг оптимизатора (SGD)

In [8]:
def sgd_update(params, grads, lr):
    conv_layers = len(params.get('_conv_layers', []))
    for i in range(1, conv_layers+1):
        params[f'W{i}'] -= lr * grads[f'W{i}']
        params[f'b{i}'] -= lr * grads[f'b{i}']
    params[f'W{conv_layers+1}'] -= lr * grads[f'W{conv_layers+1}']
    params[f'b{conv_layers+1}'] -= lr * grads[f'b{conv_layers+1}']
    return params

### Ячейка 9 — эпоха обучения

In [9]:
def train_epoch_cnn(X, y, params, batch_size, lr, augment=False):
    n = X.shape[0]; perm = np.random.permutation(n); total_loss = 0.0
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        Xb = X[idx].copy()
        yb = y[idx]
        if augment:
            for j in range(Xb.shape[0]):
                if np.random.rand() < 0.5: Xb[j] = np.flip(Xb[j], axis=1)
        yb_oh = to_one_hot(yb, num_classes)
        probs, cache = forward_batch(Xb, params)
        grads = backward_batch(yb_oh, params, cache)
        params = sgd_update(params, grads, lr)
        loss = -np.mean(np.sum(yb_oh * np.log(np.clip(probs, 1e-12, 1.0)), axis=1))
        total_loss += loss * Xb.shape[0]
    return params, total_loss / n

### Ячейка 10 — обратное распространение ошибки

In [10]:
def backward_batch(yb_onehot,params,cache):
    grads={};conv_layers=len(params.get('_conv_layers',[]));N=yb_onehot.shape[0]
    dz_mlp=(cache['probs']-yb_onehot)/N
    grads[f'W{conv_layers+1}']=cache['flat'].T@dz_mlp
    grads[f'b{conv_layers+1}']=np.sum(dz_mlp,axis=0).astype(np.float32)
    dflat=dz_mlp@params[f'W{conv_layers+1}'].T
    d_output=dflat.reshape(cache['output_shape'])
    for i in range(conv_layers,0,-1):
        cols=cache[f'cols{i}'];a_relu=cache[f'a{i}'];mask=cache[f'mask{i}']
        Nn,H_conv,W_conv,C_out=a_relu.shape
        pool=2;stride=2
        d_pool=d_output
        d_before_pool=mask*d_pool[...,None,:]
        d_before_pool=d_before_pool.reshape(Nn,d_pool.shape[1],
        d_pool.shape[2],pool,pool,C_out)
        dz_conv=np.zeros((Nn,H_conv,W_conv,C_out),dtype=np.float32)
        for p in range(pool):
            p_end=p+stride*d_pool.shape[1]
            for q in range(pool):
                q_end=q+stride*d_pool.shape[2]
                tmp = d_before_pool[:,:,:,p,q,:]
                dz_conv[:,p:p_end:stride,q:q_end:stride,:]+=tmp
        dz_conv=dz_conv*(a_relu>0)
        dz_conv_resh=dz_conv.reshape(Nn,H_conv*W_conv,C_out)
        gradW_cols=np.tensordot(cols,dz_conv_resh,axes=([0,2],[0,1]))
        W=params[f'W{i}'];k=W.shape[0];C_in=W.shape[2]
        grads[f'W{i}']=gradW_cols.reshape(k,k,C_in,C_out).astype(np.float32)
        grads[f'b{i}']=np.sum(dz_conv,axis=(0,1,2)).astype(np.float32)
        filters=W.reshape(-1,C_out)
        dcols_nlk=dz_conv_resh@filters.T; dcols=dcols_nlk.transpose(0,2,1)
        input_shape=cache[f'input_shape{i}']
        pad=params.get('_conv_layers',[{'pad':0}])[i-1].get('pad',0)
        d_input=col2im_batch_strides(dcols,input_shape,k,stride=1,padding=pad)
        d_output=d_input
    return grads

### Ячейка 11 — цикл обучения на заданное число эпох

In [11]:
def train_loop_cnn(params, X_train, y_train, X_val, y_val,
    epochs=10, batch_size=32, lr=1e-3, augment=False):
    history = {'loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for ep in range(epochs):
        params, train_loss = train_epoch_cnn(X_train, y_train, params,
        batch_size, lr, augment=augment)
        probs_val, _ = forward_batch(X_val, params)
        val_loss = to_one_hot(y_val, num_classes)*np.log(np.clip(probs_val,
        1e-12,1.0))
        val_loss = -np.mean(np.sum(val_loss, axis=1))
        sample_n = min(2000, len(X_train))
        train_probs, _ = forward_batch(X_train[:sample_n], params)
        train_acc = np.mean(np.argmax(train_probs, axis=1) == y_train[:sample_n])
        val_acc = np.mean(np.argmax(probs_val, axis=1) == y_val)
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        out = f'Эпоха {ep+1}/{epochs}: loss={train_loss:.4f}'
        out += f', val_loss={val_loss:.4f}'
        out += f', train_acc={train_acc:.4f}, val_acc={val_acc:.4f}'
        print(out)
    return params, history

### Ячейка 12 — загрузка датасета MNIST (п. 2.2)

In [12]:
class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
img_height = 14
img_width = 14
val_fraction = 0.2
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X_all = np.zeros((mnist["data"].shape[0], img_height, img_width, 1))
for i in range(mnist["data"].shape[0]):
    img_data = mnist["data"][i].reshape(28, 28)
    img = Image.fromarray(img_data.astype(np.uint8))
    resized_img = img.resize((img_width, img_height),resample=Image.LANCZOS)
    X_all[i] = np.array(resized_img).reshape(img_height, img_width, 1)
X_all = X_all.astype(np.float32) / 255.0
y_all = mnist["target"].astype(int)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all,y_all,test_size=val_fraction,random_state=RND,stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_fraction,
    random_state=RND, stratify=y_trainval
)
num_classes = len(np.unique(y_all))
print("Размеры:", X_train.shape, X_val.shape, X_test.shape)
print("Классов:", num_classes)

Размеры: (44800, 14, 14, 1) (11200, 14, 14, 1) (14000, 14, 14, 1)
Классов: 10


### Ячейка 13 — обучение свёрточной сети (п. 2.3, вариант 1)

In [13]:
epochs = 4
lr = 0.01
batch_size = 16
params = init_cnn_params(input_channels=1, num_classes=num_classes,
conv_layers=[{'k' : 3, 'out' : 16, 'pad' : 1}])
params, history = train_loop_cnn(params, X_train, y_train, X_val, y_val,
epochs=epochs, batch_size=batch_size, lr=lr, augment=True)

Эпоха 1/4: loss=1.0899, val_loss=0.6206, train_acc=0.8095, val_acc=0.7906


Эпоха 2/4: loss=0.5415, val_loss=0.4890, train_acc=0.8545, val_acc=0.8475


Эпоха 3/4: loss=0.4645, val_loss=0.4358, train_acc=0.8685, val_acc=0.8613


Эпоха 4/4: loss=0.4144, val_loss=0.4053, train_acc=0.8750, val_acc=0.8699


### Ячейка 14 — результаты предсказаний (п. 2.4)
«НС» — предсказание нейросети, «Ист.» — истинный класс.

In [14]:
rng = np.random.default_rng(123)
idxs = rng.choice(len(X_test), size=9, replace=False)
X_sel = X_test[idxs]
y_sel = y_test[idxs]
probs, _ = forward_batch(X_sel, params)
preds = np.argmax(probs, axis=1)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, i in zip(axes.flat, range(9)):
    img = X_sel[i].squeeze(); ax.imshow(img, cmap='gray')
    pr, tr = class_names[int(preds[i])], class_names[int(y_sel[i])]
    ax.set_title(f'НС:{pr} | Ист.:{tr}'); ax.axis('off')
plt.tight_layout()
plt.show()

### Ячейка 15 — сравнение с многослойным перцептроном (п. 2.5)

In [15]:
X_train_flat = X_train.reshape(-1, img_height * img_width)
X_val_flat = X_val.reshape(-1, img_height * img_width)
mlp = MLPClassifier(
hidden_layer_sizes=(int(img_height * img_width / 8), #1 скрытый
int(img_height * img_width / 16)), # 2 скрытый
max_iter=50, alpha=4e-3, solver='sgd',
random_state=RND)
mlp.fit(X_train_flat, y_train)
val_predictions=mlp.predict(X_val_flat)
val_accuracy = accuracy_score(y_val, val_predictions)
probs_v,_=forward_batch(X_val,params)
preds_v = np.argmax(probs_v, axis=1)
val_cnn=accuracy_score(y_val,preds_v)
print(f"Точность MLP на вал. выборке: {val_accuracy:.4f}")
print(f"Точность CNN на вал. выборке: {val_cnn:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Точность MLP на вал. выборке: 0.9249
Точность CNN на вал. выборке: 0.8699


## 2.6. Датасет Fashion-MNIST
### Копия ячейки 12 — загрузка Fashion-MNIST (п. 2.6.1)

In [16]:
class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']
img_height = 14
img_width = 14
val_fraction = 0.2
mnist = fetch_openml("Fashion-MNIST", version=1, as_frame=False)
X_all = np.zeros((mnist["data"].shape[0], img_height, img_width, 1))
for i in range(mnist["data"].shape[0]):
    img_data = mnist["data"][i].reshape(28, 28)
    img = Image.fromarray(img_data.astype(np.uint8))
    resized_img = img.resize((img_width, img_height),resample=Image.LANCZOS)
    X_all[i] = np.array(resized_img).reshape(img_height, img_width, 1)
X_all = X_all.astype(np.float32) / 255.0
y_all = mnist["target"].astype(int)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all,y_all,test_size=val_fraction,random_state=RND,stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_fraction,
    random_state=RND, stratify=y_trainval
)
num_classes = len(np.unique(y_all))
print("Размеры:", X_train.shape, X_val.shape, X_test.shape)
print("Классов:", num_classes)

Размеры: (44800, 14, 14, 1) (11200, 14, 14, 1) (14000, 14, 14, 1)
Классов: 10


### Копия ячейки 13 — обучение (п. 2.6.2)

In [17]:
epochs = 4
lr = 0.01
batch_size = 16
params = init_cnn_params(input_channels=1, num_classes=num_classes,
conv_layers=[{'k' : 3, 'out' : 16, 'pad' : 1}])
params, history = train_loop_cnn(params, X_train, y_train, X_val, y_val,
epochs=epochs, batch_size=batch_size, lr=lr, augment=True)

Эпоха 1/4: loss=1.0346, val_loss=0.7719, train_acc=0.7225, val_acc=0.7256


Эпоха 2/4: loss=0.6977, val_loss=0.6977, train_acc=0.7495, val_acc=0.7446


Эпоха 3/4: loss=0.6379, val_loss=0.6078, train_acc=0.7980, val_acc=0.7943


Эпоха 4/4: loss=0.5932, val_loss=0.5665, train_acc=0.8105, val_acc=0.8073


### Копии ячеек 14 и 15 — предсказания и сравнение с MLP (п. 2.6.3)

In [18]:
rng = np.random.default_rng(123)
idxs = rng.choice(len(X_test), size=9, replace=False)
X_sel = X_test[idxs]
y_sel = y_test[idxs]
probs, _ = forward_batch(X_sel, params)
preds = np.argmax(probs, axis=1)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, i in zip(axes.flat, range(9)):
    img = X_sel[i].squeeze(); ax.imshow(img, cmap='gray')
    pr, tr = class_names[int(preds[i])], class_names[int(y_sel[i])]
    ax.set_title(f'НС:{pr} | Ист.:{tr}'); ax.axis('off')
plt.tight_layout()
plt.show()

In [19]:
X_train_flat = X_train.reshape(-1, img_height * img_width)
X_val_flat = X_val.reshape(-1, img_height * img_width)
mlp = MLPClassifier(
hidden_layer_sizes=(int(img_height * img_width / 8), #1 скрытый
int(img_height * img_width / 16)), # 2 скрытый
max_iter=50, alpha=4e-3, solver='sgd',
random_state=RND)
mlp.fit(X_train_flat, y_train)
val_predictions=mlp.predict(X_val_flat)
val_accuracy = accuracy_score(y_val, val_predictions)
probs_v,_=forward_batch(X_val,params)
preds_v = np.argmax(probs_v, axis=1)
val_cnn=accuracy_score(y_val,preds_v)
print(f"Точность MLP на вал. выборке: {val_accuracy:.4f}")
print(f"Точность CNN на вал. выборке: {val_cnn:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Точность MLP на вал. выборке: 0.8219
Точность CNN на вал. выборке: 0.8073


## 3. Свёрточная сеть на цветных изображениях
Выбран датасет **«Настроение клиентов по лицам»**: лица людей в различном настроении (для оценки удовлетворённости клиентов).
### Ячейка 1 табл. 15.3 — загрузка датасета (п. 3.1–3.2)

In [20]:
link = 'https://github.com/heymynameis/Datasets-for-lab/raw/refs/heads/main/face_emotion.zip'
img_width = 64; img_height = 64; val_fraction = 0.2
if '.zip' in link:
    file_name = 'dataset.zip'; urllib.request.urlretrieve(link, file_name)
    folder_for_data = 'data'; win_path = 'C:/Users/Public/'+folder_for_data
    archive = zipfile.ZipFile(file_name)
    if 'windows' in platform.system().lower(): data_path = win_path
    else: data_path = folder_for_data
    if os.path.isdir(data_path): shutil.rmtree(data_path)
    archive.extractall(path=data_path); data_dir = last_root_folder(data_path)
else:
    file_name = link.split('/')[-1]
    data_dir = utils.get_file(file_name, origin=link, extract=True)
    data_dir = last_root_folder(data_dir)
pics = ['.jpg','.jpeg','.webp','.png']; last_img = ''; num_of_img = 0
for path, subdirs, files in os.walk(data_dir):
    for name in files:
        if any([ext in os.path.join(path, name) for ext in pics]):
            num_of_img+=1; last_img = os.path.join(path, name)
print(f'В датасете {num_of_img} изображений')
if last_img:
    try:
        with Image.open(str(last_img)) as one_of_imgs:
            print(f'Исходный размер: {one_of_imgs.size[0]}x{one_of_imgs.size[1]}')
    except Exception as e:
        print(f"Не удалось открыть {last_img}: {e}")
images, labels = [], []
class_names = sorted([
d for d in os.listdir(data_dir)
if os.path.isdir(os.path.join(data_dir, d))])
label_to_int = {name: i for i, name in enumerate(class_names)}
int_to_label = {i: name for i, name in enumerate(class_names)}
num_classes = len(class_names)
print(f"Найденные классы: {class_names}")
for class_name in class_names:
    class_path = os.path.join(data_dir, class_name)
    if not os.path.isdir(class_path): continue
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        try:
            with Image.open(img_path) as img:
                img=img.resize((img_width,img_height)).convert("RGB")
                images.append(np.array(img))
                labels.append(label_to_int[class_name])
        except Exception as e: print(f"Не удалось прочитать {img_path}: {e}")

В датасете 2100 изображений
Исходный размер: 128x128
Найденные классы: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


### Ячейка 2 табл. 15.3 — подготовка данных (п. 3.3)

In [21]:
X_all = np.array(images)
y_all = np.array(labels)
X_all = X_all.reshape(-1, img_height, img_width, 3).astype(np.float32) / 255.0
X_trainval, X_test, y_trainval, y_test = train_test_split(
X_all, y_all, test_size=val_fraction, random_state=RND, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
X_trainval, y_trainval, test_size=val_fraction, random_state=RND,
stratify=y_trainval)
print("Размеры:", X_train.shape, X_val.shape, X_test.shape)
print("Классов:", num_classes)

Размеры: (1344, 64, 64, 3) (336, 64, 64, 3) (420, 64, 64, 3)
Классов: 7


### Копия ячейки 13 — обучение на цветных изображениях (п. 3.4–3.5)
Число эпох: 4 × n = 4 × 3 = 12, `input_channels = 3`.

In [22]:
epochs = 12
lr = 0.01
batch_size = 16
params = init_cnn_params(input_channels=3, num_classes=num_classes,
conv_layers=[{'k' : 3, 'out' : 16, 'pad' : 1}])
params, history = train_loop_cnn(params, X_train, y_train, X_val, y_val,
epochs=epochs, batch_size=batch_size, lr=lr, augment=True)

Эпоха 1/12: loss=1.9556, val_loss=1.9406, train_acc=0.1793, val_acc=0.1667


Эпоха 2/12: loss=1.9331, val_loss=1.9306, train_acc=0.2009, val_acc=0.1696


Эпоха 3/12: loss=1.9221, val_loss=1.9235, train_acc=0.2128, val_acc=0.1815


Эпоха 4/12: loss=1.9091, val_loss=1.9106, train_acc=0.2336, val_acc=0.2083


Эпоха 5/12: loss=1.8915, val_loss=1.9251, train_acc=0.2299, val_acc=0.2143


...

Эпоха 8/12: loss=1.8053, val_loss=1.8305, train_acc=0.3162, val_acc=0.2500


Эпоха 9/12: loss=1.7822, val_loss=1.8215, train_acc=0.3244, val_acc=0.2768


Эпоха 10/12: loss=1.7599, val_loss=1.7811, train_acc=0.3348, val_acc=0.3125


Эпоха 11/12: loss=1.7238, val_loss=1.7749, train_acc=0.3400, val_acc=0.3095


Эпоха 12/12: loss=1.6825, val_loss=1.7626, train_acc=0.3579, val_acc=0.3214


### Копия ячейки 14 — предсказания в цвете (п. 3.6)

In [23]:
rng = np.random.default_rng(123)
idxs = rng.choice(len(X_test), size=9, replace=False)
X_sel = X_test[idxs]
y_sel = y_test[idxs]
probs, _ = forward_batch(X_sel, params)
preds = np.argmax(probs, axis=1)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, i in zip(axes.flat, range(9)):
    img = X_sel[i].squeeze(); ax.imshow(img)
    pr, tr = class_names[int(preds[i])], class_names[int(y_sel[i])]
    ax.set_title(f'НС:{pr} | Ист.:{tr}'); ax.axis('off')
plt.tight_layout()
plt.show()

### Копия ячейки 15 — сравнение с MLP для цветных изображений (п. 3.7)

In [24]:
X_train_flat = X_train.reshape(-1, img_height * img_width * 3)
X_val_flat = X_val.reshape(-1, img_height * img_width * 3)
mlp = MLPClassifier(
hidden_layer_sizes=(int(img_height * img_width * 3 / 8), #1 скрытый
int(img_height * img_width / 16)), # 2 скрытый
max_iter=50, alpha=4e-3, solver='sgd',
random_state=RND)
mlp.fit(X_train_flat, y_train)
val_predictions=mlp.predict(X_val_flat)
val_accuracy = accuracy_score(y_val, val_predictions)
probs_v,_=forward_batch(X_val,params)
preds_v = np.argmax(probs_v, axis=1)
val_cnn=accuracy_score(y_val,preds_v)
print(f"Точность MLP на вал. выборке: {val_accuracy:.4f}")
print(f"Точность CNN на вал. выборке: {val_cnn:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Точность MLP на вал. выборке: 0.4405
Точность CNN на вал. выборке: 0.3214


## 4. Аугментация изображений
### Ячейка 1 табл. 15.4 — аугментация одного изображения

In [25]:
def augment_image(arr, rot=15, crop=0.12, noise=0.05):
    if arr.ndim == 2: channels = 1; a = (arr * 255).astype(np.uint8)
    else: channels = arr.shape[2]; a = (arr * 255).astype(np.uint8)
    img = Image.fromarray(a)
    angle = np.random.uniform(-rot, rot)
    img = img.rotate(angle, resample=Image.BILINEAR)
    if crop > 0:
        w, h = img.size
        cw = int(w * (1 - crop)); ch = int(h * (1 - crop))
        if cw < 1: cw = 1
        if ch < 1: ch = 1
        left = np.random.randint(0, w - cw + 1)
        top = np.random.randint(0, h - ch + 1)
        img = img.crop((left, top, left + cw, top + ch)).resize((w, h),
        Image.BILINEAR)
    if np.random.rand() < 0.5: img = ImageOps.mirror(img)
    arr2 = np.array(img, dtype=np.float32) / 255.0
    if noise > 0:
        arr2 = arr2 + np.random.randn(*arr2.shape) * noise
        arr2 = np.clip(arr2, 0.0, 1.0)
    if channels == 1 and arr2.ndim == 2:
        return arr2.reshape(arr2.shape[0], arr2.shape[1], 1)
    return arr2.astype(np.float32)

### Ячейка 2 табл. 15.4 — аугментация всего датасета

In [26]:
def augment_dataset(X, y, factor=1, rot=15, crop=0.12, noise=0.05,
    keep_original=True):
    if factor <= 0:
        return X.copy(), y.copy()
    X_list = []; y_list = []; is_color = X.ndim == 4
    if keep_original:
        X_list.append(X.copy()); y_list.append(np.asarray(y).copy())
    N = X.shape[0]
    for i in range(N):
        for _ in range(factor):
            aug = augment_image(X[i], rot=rot, crop=crop, noise=noise)
            if is_color: X_list.append(np.expand_dims(aug, 0))
            else: X_list.append(np.expand_dims(np.expand_dims(aug, 0), -1))
            y_list.append(np.array([y[i]]))
    X_out = np.concatenate(X_list, axis=0)
    y_out = np.concatenate(y_list, axis=0)
    return X_out.astype(np.float32), y_out.astype(int)

### Ячейка 3 табл. 15.4 — визуализация аугментации

In [27]:
def visualize_augmentations(X, y, n_samples=9, n_aug=4, seed=123):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(X), size=n_samples, replace=False)
    X_sel = X[idxs]
    y_sel = y[idxs]
    fig, axes = plt.subplots(n_samples, n_aug + 1, figsize=(2*(n_aug+1),
    2*n_samples))
    for i in range(n_samples):
        orig = X_sel[i].squeeze()
        axes[i,0].imshow(orig)
        axes[i,0].set_title(f"Класс {class_names[int(y_sel[i])]}")
        axes[i,0].axis('off')
        for j in range(n_aug):
            aug = augment_image(X_sel[i])
            axes[i,j+1].imshow(aug.squeeze())
            axes[i,j+1].axis('off')
    plt.tight_layout()
    plt.show()

### Ячейка 4 табл. 15.4 — аугментация датасета и примеры

In [28]:
X_train_aug, y_train_aug = augment_dataset(X_train, y_train,
factor=1, rot=15, crop=0.12, noise=0.05, keep_original=True)
visualize_augmentations(X_test, y_test, n_samples=9, n_aug=4, seed=42)

### Ячейка 5 табл. 15.4 — дообучение сети на аугментированных данных (5 эпох)

In [29]:
params, history_aug = train_loop_cnn(params, X_train_aug, y_train_aug, X_val,
y_val, epochs=5, batch_size=batch_size, lr=lr, augment=False)

Эпоха 1/5: loss=1.7395, val_loss=1.7035, train_acc=0.3770, val_acc=0.3482


Эпоха 2/5: loss=1.6961, val_loss=1.7012, train_acc=0.3845, val_acc=0.3571


Эпоха 3/5: loss=1.6578, val_loss=1.6881, train_acc=0.3930, val_acc=0.3750


Эпоха 4/5: loss=1.6228, val_loss=1.6878, train_acc=0.4015, val_acc=0.3690


Эпоха 5/5: loss=1.5819, val_loss=1.6624, train_acc=0.4220, val_acc=0.3631


### Копия ячейки из п. 3.7 — сравнение с MLP после аугментации (п. 4.2)

In [30]:
X_train_flat = X_train.reshape(-1, img_height * img_width * 3)
X_val_flat = X_val.reshape(-1, img_height * img_width * 3)
mlp = MLPClassifier(
hidden_layer_sizes=(int(img_height * img_width * 3 / 8), #1 скрытый
int(img_height * img_width / 16)), # 2 скрытый
max_iter=50, alpha=4e-3, solver='sgd',
random_state=RND)
mlp.fit(X_train_flat, y_train)
val_predictions=mlp.predict(X_val_flat)
val_accuracy = accuracy_score(y_val, val_predictions)
probs_v,_=forward_batch(X_val,params)
preds_v = np.argmax(probs_v, axis=1)
val_cnn=accuracy_score(y_val,preds_v)
print(f"Точность MLP на вал. выборке: {val_accuracy:.4f}")
print(f"Точность CNN на вал. выборке: {val_cnn:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Точность MLP на вал. выборке: 0.4405
Точность CNN на вал. выборке: 0.3631


## 5. Модель, обученная большим числом эпох
Готовая модель для датасета «Настроение клиентов по лицам» (обучалась 325 эпох, архитектура по табл. 15.5).
### Ячейка 1 табл. 15.6 — загрузка весов (п. 5.1)

In [31]:
weights_url = 'https://github.com/heymynameis/Datasets-for-lab/raw/refs/heads/main/20_faces_325.npy'
response = requests.get(weights_url)
if response.status_code == 200:
    file_like_object = io.BytesIO(response.content)
    params = np.load(file_like_object, allow_pickle=True).item()
    print("Параметры модели загружены в params")
    print(f"Имена параметров модели: {params.keys()}")
else: print(f"Ошибка загрузки: статус код {response.status_code}")

Параметры модели загружены в params
Имена параметров модели: dict_keys(['W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'W4', 'b4', '_conv_layers', '_input_size'])


### Ячейка 2 табл. 15.6 — визуализация ядер свёртки (п. 5.2)

In [32]:
def visualize_conv_kernels(weights_array, layer_name, max_kernels_to_show=32):
    W = weights_array.squeeze()
    if W.ndim == 4: kernel_h, kernel_w, C_in, C_out = W.shape
    else: print(f"!!Размерность {W.ndim} для {layer_name}");return
    num_kernels_to_plot = min(C_out, max_kernels_to_show)
    rows = int(np.ceil(np.sqrt(num_kernels_to_plot)))
    cols = int(np.ceil(num_kernels_to_plot / rows))
    fig, axes = plt.subplots(rows, cols, figsize=(5, 5))
    if C_in > 3:
        fig.suptitle(f'Ядра свертки {layer_name}', fontsize=12)
        for i, ax in enumerate(axes.flat):
            if i < num_kernels_to_plot:
                kernel = W[:, :, :, i]; kernel_mean = np.mean(kernel, axis=2)
                k_min, k_max = kernel_mean.min(), kernel_mean.max()
                if k_max-k_min!=0:kernel_norm=(kernel_mean-k_min)/(k_max-k_min)
                else: kernel_norm = kernel_mean * 0.0
                ax.imshow(kernel_norm, cmap='gray')
                ax.set_title(f'Ядро {i+1}', fontsize=8)
                ax.axis('off')
            else: ax.axis('off')
    else:
        fig.suptitle(f'Ядра свертки: {layer_name}', fontsize=12)
        for i, ax in enumerate(axes.flat):
            if i < num_kernels_to_plot:
                kernel = W[:, :, :, i]
                kernel_norm=(kernel-kernel.min())/(kernel.max()-kernel.min())
                if C_in == 1: ax.imshow(kernel_norm.squeeze(), cmap='gray')
                else: ax.imshow(kernel_norm)
                ax.set_title(f'Ядро {i+1}', fontsize=8)
                ax.axis('off')
            else: ax.axis('off')
    plt.tight_layout(rect=[0, 0, 1, 1])
    plt.show()
visualize_conv_kernels(params['W1'], 'W1')
visualize_conv_kernels(params['W2'], 'W2')
visualize_conv_kernels(params['W3'], 'W3')

### Предсказания заранее обученной модели (п. 5.3)

In [33]:
rng = np.random.default_rng(123)
idxs = rng.choice(len(X_test), size=9, replace=False)
X_sel = X_test[idxs]
y_sel = y_test[idxs]
probs, _ = forward_batch(X_sel, params)
preds = np.argmax(probs, axis=1)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, i in zip(axes.flat, range(9)):
    img = X_sel[i].squeeze(); ax.imshow(img)
    pr, tr = class_names[int(preds[i])], class_names[int(y_sel[i])]
    ax.set_title(f'НС:{pr} | Ист.:{tr}'); ax.axis('off')
plt.tight_layout()
plt.show()

### Сравнение заранее обученной модели с MLP (п. 5.3)

In [34]:
X_train_flat = X_train.reshape(-1, img_height * img_width * 3)
X_val_flat = X_val.reshape(-1, img_height * img_width * 3)
mlp = MLPClassifier(
hidden_layer_sizes=(int(img_height * img_width * 3 / 8), #1 скрытый
int(img_height * img_width / 16)), # 2 скрытый
max_iter=50, alpha=4e-3, solver='sgd',
random_state=RND)
mlp.fit(X_train_flat, y_train)
val_predictions=mlp.predict(X_val_flat)
val_accuracy = accuracy_score(y_val, val_predictions)
probs_v,_=forward_batch(X_val,params)
preds_v = np.argmax(probs_v, axis=1)
val_cnn=accuracy_score(y_val,preds_v)
print(f"Точность MLP на вал. выборке: {val_accuracy:.4f}")
print(f"Точность CNN на вал. выборке: {val_cnn:.4f}")

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Точность MLP на вал. выборке: 0.4405
Точность CNN на вал. выборке: 0.6042


## Выводы

1. **MNIST (цифры, 14×14).** Свёрточная сеть варианта 1 (один свёрточный слой: 16 ядер 3×3, отступ 1, пулинг 2×2) за 4 эпохи достигла точности **0,870** на валидационной выборке; loss падал от эпохи к эпохе (1,09 → 0,41). MLP показал **0,925**. При одном свёрточном слое и всего 4 эпохах CNN не успевает обогнать перцептрон на таком простом датасете.
2. **Fashion-MNIST (одежда, 14×14).** Задача сложнее, точность ниже: CNN — **0,807**, MLP — **0,822**. Ошибки в основном на похожих классах (рубашка / футболка / пуловер).
3. **Цветные изображения — «Настроение клиентов по лицам» (64×64×3, 7 классов, 2100 изображений).** За 12 эпох CNN получила **0,321** (случайное угадывание — 0,143), MLP — **0,440**. Модель намеренно недообучена: точность растёт от эпохи к эпохе, но эпох мало.
4. **Аугментация** (поворот, обрезка, зеркальное отражение, шум) удвоила обучающую выборку (1344 → 2688). После дообучения на 5 эпох точность CNN выросла с **0,321 до 0,363**.
5. **Заранее обученная модель** (3 свёрточных слоя, 325 эпох, табл. 15.5) дала **0,604** — заметно лучше самостоятельно обученной модели (0,363) и MLP (0,440). Ядра первого слоя похожи на детекторы перепадов яркости (границ, углов); на 2-м и 3-м слоях после усреднения ядер признаки менее наглядны.

**Итог:** свёрточные сети сами извлекают признаки из изображений. Чтобы их преимущество проявилось, нужны более глубокая архитектура и достаточное число эпох. Аугментация помогает при малом объёме данных.